In [2]:
with open("dataset.txt") as f:
    text = f.read()

In [3]:
len(text)

1115394

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = { ch:i for i, ch in enumerate(chars) }
itos = { i:ch for i, ch in enumerate(chars) }
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: "".join([itos[i] for i in l])
print(encode("hello"))
print(decode(encode("hello")))

#Sentence Piece, tiktoken

[46, 43, 50, 50, 53]
hello


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [7]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input is {context} and target is {target}")

input is tensor([18]) and target is 47
input is tensor([18, 47]) and target is 56
input is tensor([18, 47, 56]) and target is 57
input is tensor([18, 47, 56, 57]) and target is 58
input is tensor([18, 47, 56, 57, 58]) and target is 1
input is tensor([18, 47, 56, 57, 58,  1]) and target is 15
input is tensor([18, 47, 56, 57, 58,  1, 15]) and target is 47
input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target is 58


In [10]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8  

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print('input:')
print(xb.shape)
print(xb)
print('target:')
print(yb.shape)
print(yb)

print("------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"input is {context} and target is {target}")

input:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------
input is tensor([24]) and target is 43
input is tensor([24, 43]) and target is 58
input is tensor([24, 43, 58]) and target is 5
input is tensor([24, 43, 58,  5]) and target is 57
input is tensor([24, 43, 58,  5, 57]) and target is 1
input is tensor([24, 43, 58,  5, 57,  1]) and target is 46
input is tensor([24, 43, 58,  5, 57,  1, 46]) and target is 43
input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) and target is 39
input is tensor([44]) and target is 53
input is tensor([44, 53]) and target is 56
input is tensor([44, 53, 56]) and target is 1
input is tensor([44, 53, 56,  1]) and target is 58
i

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

class Bigram(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # [B,T,C]
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            # targets = targets.view(-1) same as below
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is [B,T] 
        for _ in range(max_new_tokens):
            # get the prediction
            logits, loss = self(idx)
            # focus on the last time step
            logits = logits[:,-1,:] # becmoes [B,C]
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # [B,1]
            idx = torch.cat((idx, idx_next), dim=1)  # [B,T+1]
        return idx
        

m = Bigram(vocab_size)
# logits, loss = m(xb, yb) #this gives because pytorch needs B/C/T NOT B/T/C
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) #-ln(1/65) = 4.17

print(decode(m.generate(idx= torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [12]:
optimser = torch.optim.Adam(m.parameters(), lr=1e-3)

In [13]:
batch_size = 32
for steps in range(5000):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimser.zero_grad()
    loss.backward()
    optimser.step()

print(loss.item())  

2.6783480644226074


In [14]:
print(decode(m.generate(idx= torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Mave ghtang d ws amangT:
D: nd TINT:
Fof;AURKIf diTus, be:ghere
Dyo.O'd, t fed inks cerVis benes. owepwnof pre, athar wowir W:
ALI u
INCAbOgfatho rend thic; is be wara!
PO.
WGListh R.
JaveauCE! ce, ince:'
t.
Sin honk$zlerseestindovrer wat boue nodgh flle, mOM:

SCKpoto AKIFwinthind d me sesete gkerw'DWhe arDERe

HET:
ROLINanxueile to he bis wllagGomieg.
DWhelat s tropghallll,
Wh trnenQJMmllel cLENzt iBOFokKENor-mu ed de atos are th, a!OFot I s l:
E-de pond
TII'henJAn?Bof bogave tthe s me : t
Yo-


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Maths trick in self attention

In [16]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [17]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # xbow is x bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # [t,C]
        xbow[b,t] = torch.mean(xprev, 0)
        

In [18]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [19]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

## above all are inequicent

In [20]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a/ torch.sum(a, dim=1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print("a:", a)
print("b:", b)
print("c:", c)


a: tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b: tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c: tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


## Coontine

## version was without matrix
## version2

In [21]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x # (T, T) @ (B, T, C) ---- torch makes it (B, T, T) @ (B, T, C) = (B, T, C)
torch.allclose(xbow, xbow2) # should give True maybe some update in torch

False

## version 3
run one by one if u dont understand

## We will use this

In [22]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T) #interaction strength
wei = wei.masked_fill(tril==0, float('-inf')) #tokens form past cant commmnunicate(basically future cant communicate with past)
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x # aggregation
torch.allclose(xbow, xbow3) #again 

False

## version 4: Self attention
v3 has uniform numbers because different tokens find other interesting or not so to make it data dependent we will use self attention

In [23]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16  
key = nn.Linear(C, head_size, bias=False) # what do i contain
query = nn.Linear(C, head_size, bias=False) # what am i looking for
value = nn.Linear(C, head_size, bias=False) # what do i want to know
k = key(x) # [B,T,16]
q = query(x) # [B,T,16]
wei = q @ k.transpose(-2, -1) # [B,T,16] @ [B,16,T] = [B,T,T]


tril = torch.tril(torch.ones(T,T))
# wei = torch.zeros(T,T)
wei = wei.masked_fill(tril==0, float('-inf')) # if u remove this line they will completefy communicate but if we keep it will communicate only in one direction i.e. past to future Ex: Sentimental analysis it needs future also
wei = F.softmax(wei, dim=-1)

v = value(x) # [B,T,16]
out = wei @ v # [B,T,T] @ [B,T,16] = [B,T,16]


# out = wei @ x
out.shape

torch.Size([4, 8, 16])

In [24]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [25]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2,-1) * head_size**-0.5 #scaled attention(this is done to preserve variance which is close to 1 shown below)

In [27]:
k.var() 
q.var()
wei.var() #before scaling it was 17.46 which is way greater than head_size = 16

tensor(1.0918)

## check batch norm code in make more and see layer norm